# Example experiment on the controller

The general flow a workflow follows on top of the controller. It talks only to
the controller - never a vendor API - so the same notebook runs against any
microscope that has a driver.

1. connect - pick an instrument and set the frame
2. select states - capture a configuration per scan mode
3. get positions - the positions to visit
4. acquire - set a mode, then for each position: set_xyz, acquire (captures + saves)

## Setup

Register the bundled **mock driver** so the flow runs with no hardware. Run this
notebook with its own folder as the working directory.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()  # .../zmart_controller
sys.path.insert(0, str(here.parents[0]))  # repo root (parent of the package)
sys.path.insert(0, str(here / "tests"))  # the mock driver

import mock_driver  # noqa: E402

mock_driver.register_mock()

## 1. Connect

`get_instruments()` lists what you can connect to as connection dicts (touching no
hardware). Pick one, and `set_instrument()` opens the session. The driver loads
its own configuration as it connects, including the origin: the (0, 0, 0) point
that every position is measured from, in micrometers. You set that origin once,
in the driver's own setup, not in the experiment. From here, `zmart_controller`
is that microscope.

In [ ]:
import zmart_controller

zmart_controller.get_instruments()

In [ ]:
instrument = zmart_controller.get_instruments()[0]
zmart_controller.set_instrument(instrument)

## 2. Select states

A **state** is a snapshot of the instrument you can reapply later: a `changeable`
part (exactly what `set_state` reapplies) plus a read-only `observed` report. It
is opaque to the controller - only the driver interprets it. We capture two
modes, one cell each.


In [ ]:
# a gentle overview scan
prescan = zmart_controller.get_state()
prescan["changeable"]["laser_power"] = 2.0
prescan

In [ ]:
# a higher-power detailed scan
target = zmart_controller.get_state()
target["changeable"]["laser_power"] = 8.0
target["changeable"]["gain"] = 2.0
target

## 3. Choose the positions

The positions to visit are part of your experiment, so the notebook defines
them itself, in micrometers from the microscope's configured origin. This keeps the
experiment the same on every microscope. Some drivers can also report
positions drawn in the vendor's own software through `get_info()`, but that is
an extra of those drivers and not something every microscope offers.

In [ ]:
positions = [
    {"x": 0.0, "y": 0.0, "z": 0.0},
    {"x": 120.0, "y": 0.0, "z": 0.0},
    {"x": 0.0, "y": 120.0, "z": 0.0},
]
positions

## 4. Acquire

Reapply a state, then visit every position. `set_xyz` moves in micrometers from
the origin, using each axis's default motor, which every microscope has.
`acquire` captures **and** saves in one call: `acquisition_type` is the kind of
scan, and `position_label` is the name used for the saved files. Re-run the two
cells below with `zmart_controller.set_state(target)` for the detailed scan.

In [ ]:
zmart_controller.set_state(prescan)

In [ ]:
for i, pos in enumerate(positions):
    zmart_controller.set_xyz(pos["x"], pos["y"], pos["z"])
    record = zmart_controller.acquire(acquisition_type="prescan", position_label=f"pos_{i:03d}")
    print(record)

## Done

Close the session. The mock has nothing to release, but a real driver may need
the teardown.

---

*Author: Thom de Hoog — Center for Microscopy and Image Analysis (ZMB),
University of Zurich (thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com).*

In [ ]:
zmart_controller.disconnect()